# TP03: Perfilado de Datos
## Laboratorio (Herramientas) - Universidad del Aconcagua
### Unidad 2: Herramientas en la Nube de Visualización de Datos

---

### 🎯 Objetivos del Trabajo Práctico

1. Utilizar herramientas de **profiling integradas** en Databricks
2. Reconocer **distribuciones iniciales** del dataset
3. Identificar **valores atípicos y anomalías**
4. Generar **estadísticas descriptivas** automáticas
5. Visualizar **correlaciones** entre variables

---

### 📁 Caso de Estudio: Perfilado de Ventas de Panadería

Utilizaremos las herramientas nativas de Databricks para perfilar y analizar los datos de ventas.

### 🕰️ Duración Estimada: 2 horas

In [0]:
# Configurar rutas del proyecto
import os
from pathlib import Path

# Obtener usuario actual automáticamente desde Spark
# Esto hace que el notebook sea 100% portable entre usuarios
USUARIO = spark.sql("SELECT current_user()").collect()[0][0]
BASE_USER = Path(f"/Workspace/Users/{USUARIO}")
BASE_LABORATORIO = BASE_USER / "Laboratorio"
BASE_DATASETS = BASE_LABORATORIO / "05 - Datasets"

print(f"📍 Usuario: {USUARIO}")
print(f"\n📂 Rutas configuradas:")
print(f"  Usuario:     {BASE_USER}")
print(f"  Laboratorio: {BASE_LABORATORIO}")
print(f"  Datasets:    {BASE_DATASETS}")

# Verificar que las rutas existen
if BASE_DATASETS.exists():
    archivos_csv = len([f for f in os.listdir(BASE_DATASETS) if f.endswith('.csv')])
    print(f"\n✅ Rutas verificadas correctamente")
    print(f"   Archivos CSV encontrados: {archivos_csv}")
else:
    print(f"\n⚠️ Advertencia: La carpeta de datasets no existe")

In [0]:
# Importar librerías necesarias
import pandas as pd  # Manipulación y análisis de datos tabulares
import numpy as np   # Operaciones numéricas y arrays
import matplotlib.pyplot as plt  # Visualizaciones estáticas
import seaborn as sns  # Visualizaciones estadísticas avanzadas

# Configurar estilo de visualizaciones
# set_style() define el tema visual de los gráficos
sns.set_style('whitegrid')  # Fondo blanco con cuadrícula gris
# rcParams configura parámetros globales de matplotlib
plt.rcParams['figure.figsize'] = (12, 6)  # Tamaño predeterminado: 12x6 pulgadas

print("✅ Librerías importadas correctamente")
print("\n🎨 Configuración de visualizaciones lista")

## Parte 1: Carga y Preparación de Datos

### 📂 Cargar los datasets de la panadería

Vamos a cargar los datasets y preparar un DataFrame consolidado para el perfilado.

In [0]:
# Cargar todos los datasets usando las rutas configuradas
# BASE_DATASETS fue definida en la celda anterior con pathlib
# pd.read_csv() lee archivos CSV y los convierte en DataFrames

# Cargar cada dataset
df_productos = pd.read_csv(BASE_DATASETS / 'productos.csv')
df_sucursales = pd.read_csv(BASE_DATASETS / 'sucursales.csv')
df_clientes = pd.read_csv(BASE_DATASETS / 'clientes.csv')
# parse_dates=['fecha'] convierte automáticamente la columna 'fecha' a tipo datetime
df_ventas = pd.read_csv(BASE_DATASETS / 'ventas.csv', parse_dates=['fecha'])
df_detalles_ventas = pd.read_csv(BASE_DATASETS / 'detalles_ventas.csv')

print("✅ Todos los datasets cargados")
print(f"\n📊 Registros por dataset:")
# len() cuenta la cantidad de filas en cada DataFrame
print(f"  Productos: {len(df_productos):,}")
print(f"  Sucursales: {len(df_sucursales):,}")
print(f"  Clientes: {len(df_clientes):,}")
print(f"  Ventas: {len(df_ventas):,}")
print(f"  Detalles de ventas: {len(df_detalles_ventas):,}")

In [0]:
# Crear un dataset consolidado uniendo ventas con detalles y productos
# Este dataset combina toda la información relevante para el perfilado

# Paso 1: Unir detalles_ventas con productos
# .merge() combina DataFrames por una columna común (como JOIN en SQL)
# suffixes=('_detalle', '_producto') diferencia columnas con el mismo nombre
df_consolidado = df_detalles_ventas.merge(
    df_productos[['producto_id', 'nombre', 'categoria', 'precio_unitario', 'costo_unitario']],
    on='producto_id',  # Columna común para unir
    how='left',  # LEFT JOIN: mantiene todos los detalles de venta
    suffixes=('_detalle', '_producto')
).merge(
    # Paso 2: Agregar información de la venta (fecha, sucursal, cliente)
    df_ventas[['venta_id', 'fecha', 'sucursal_id', 'cliente_id', 'total']],
    on='venta_id',
    how='left'
)

# Calcular métricas adicionales derivadas
# Ganancia = Ingresos - Costos
df_consolidado['ganancia'] = df_consolidado['subtotal'] - (df_consolidado['cantidad'] * df_consolidado['costo_unitario'])
# Extraer componentes temporales de la fecha
df_consolidado['mes'] = df_consolidado['fecha'].dt.month  # Número de mes (1-12)
df_consolidado['dia_semana'] = df_consolidado['fecha'].dt.dayofweek  # 0=Lunes, 6=Domingo
df_consolidado['es_fin_semana'] = df_consolidado['dia_semana'] >= 5  # Sábado(5) o Domingo(6)

print("✅ Dataset consolidado creado")
print(f"\n📊 Total de registros: {len(df_consolidado):,}")
print(f"📌 Columnas: {len(df_consolidado.columns)}")
print(f"\n📄 Primeras filas:")
display(df_consolidado.head())

## Parte 2: Perfilado Estadístico Automático

### 📊 Análisis descriptivo completo

Vamos a analizar las características estadísticas de nuestros datos numéricos y categóricos.

In [0]:
# Seleccionar solo columnas numéricas
# .select_dtypes() filtra columnas por tipo de dato
# include=[np.number] incluye todos los tipos numéricos (int, float, etc.)
columnas_numericas = df_consolidado.select_dtypes(include=[np.number]).columns.tolist()

print("🔢 PERFIL DE VARIABLES NUMÉRICAS")
print("=" * 80)
print(f"\nVariables numéricas identificadas: {len(columnas_numericas)}")
print(f"Columnas: {columnas_numericas}")

print("\n📈 ESTADÍSTICAS DESCRIPTIVAS:")
print("=" * 80)

# Estadísticas descriptivas completas
# .describe() calcula: count, mean, std, min, 25%, 50%, 75%, max
# .T transpone el DataFrame (columnas → filas, filas → columnas)
estadisticas = df_consolidado[columnas_numericas].describe().T
# Agregar columnas de valores faltantes
estadisticas['missing'] = df_consolidado[columnas_numericas].isnull().sum()  # Cantidad de nulos
estadisticas['missing_pct'] = (estadisticas['missing'] / len(df_consolidado) * 100).round(2)  # Porcentaje

display(estadisticas)

In [0]:
# Seleccionar columnas categóricas
# include=['object', 'bool'] filtra por tipos de texto y booleanos
columnas_categoricas = df_consolidado.select_dtypes(include=['object', 'bool']).columns.tolist()

print("🏷️ PERFIL DE VARIABLES CATEGÓRICAS")
print("=" * 80)

# Iterar sobre las primeras 5 columnas categóricas
for col in columnas_categoricas[:5]:  # Mostrar las primeras 5
    # .nunique() cuenta cuántos valores diferentes (distintos) hay
    valores_unicos = df_consolidado[col].nunique()
    valores_nulos = df_consolidado[col].isnull().sum()
    
    print(f"\n📌 {col}:")
    print(f"  Valores únicos: {valores_unicos}")
    print(f"  Valores nulos: {valores_nulos} ({valores_nulos/len(df_consolidado)*100:.2f}%)")
    print(f"  Top 5 valores más frecuentes:")
    # .value_counts() cuenta cuántas veces aparece cada valor
    # .head() muestra los primeros 5 más frecuentes
    print(df_consolidado[col].value_counts().head())

## Parte 3: Visualizaciones de Distribución

### 📉 Histogramas y boxplots

Las visualizaciones nos ayudan a entender la distribución de los datos y detectar valores atípicos.

In [0]:
# Visualizar la distribución de subtotales
# Creamos una figura con 2 subgráficos lado a lado
# plt.subplots(filas, columnas, figsize=(ancho, alto))
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histograma - muestra la frecuencia de valores en rangos (bins)
# bins=50 divide el rango en 50 intervalos
# edgecolor='black' pone bordes negros a las barras
# alpha=0.7 establece transparencia (0=transparente, 1=opaco)
axes[0].hist(df_consolidado['subtotal'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('📉 Distribución de Subtotales', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Subtotal ($)')
axes[0].set_ylabel('Frecuencia')
axes[0].grid(axis='y', alpha=0.3)  # Cuadrícula solo en eje Y

# Box plot - muestra mediana, cuartiles y valores atípicos
# vert=True orienta el box plot verticalmente
axes[1].boxplot(df_consolidado['subtotal'], vert=True)
axes[1].set_title('📦 Box Plot de Subtotales', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Subtotal ($)')
axes[1].grid(axis='y', alpha=0.3)

# tight_layout() ajusta automáticamente el espaciado entre subgráficos
plt.tight_layout()
plt.show()

# Calcular estadísticas descriptivas clave
print(f"\n📊 Estadísticas de Subtotales:")
print(f"  Media: ${df_consolidado['subtotal'].mean():,.2f}")  # Promedio
print(f"  Mediana: ${df_consolidado['subtotal'].median():,.2f}")  # Valor central
print(f"  Desviación estándar: ${df_consolidado['subtotal'].std():,.2f}")  # Dispersión

In [0]:
# Distribución de cantidades vendidas
# Analiza cuántas unidades de cada producto se venden típicamente
fig, ax = plt.subplots(figsize=(12, 6))

# .value_counts() cuenta cuántas veces aparece cada cantidad (1, 2, 3, etc.)
# .sort_index() ordena por cantidad (de menor a mayor)
# kind='bar' crea un gráfico de barras verticales
df_consolidado['cantidad'].value_counts().sort_index().plot(
    kind='bar', 
    ax=ax, 
    color='skyblue', 
    edgecolor='black'
)
ax.set_title('📐 Distribución de Cantidades Vendidas', fontsize=14, fontweight='bold')
ax.set_xlabel('Cantidad de unidades')
ax.set_ylabel('Frecuencia')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Mostrar estadísticas descriptivas de las cantidades
print(f"\n📊 Estadísticas de Cantidades:")
print(df_consolidado['cantidad'].describe())

In [0]:
# Gráfico de barras: ventas por categoría
# .groupby() agrupa por categoría, .sum() suma todos los subtotales
# .sort_values(ascending=True) ordena de menor a mayor
ventas_categoria = df_consolidado.groupby('categoria')['subtotal'].sum().sort_values(ascending=True)

# Crear figura y eje
fig, ax = plt.subplots(figsize=(12, 6))
# kind='barh' crea barras horizontales (h = horizontal)
# color='coral' establece el color de las barras
ventas_categoria.plot(kind='barh', ax=ax, color='coral', edgecolor='black')
ax.set_title('🎨 Facturación Total por Categoría', fontsize=14, fontweight='bold')
ax.set_xlabel('Facturación Total ($)')
ax.set_ylabel('Categoría')
ax.grid(axis='x', alpha=0.3)  # Cuadrícula solo en eje X

# Agregar valores numéricos al final de cada barra
# enumerate() itera con índice (i) y valor (v)
for i, v in enumerate(ventas_categoria):
    # ax.text(x, y, texto) coloca texto en coordenadas (x, y)
    # va='center' alinea verticalmente al centro
    ax.text(v, i, f' ${v:,.0f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## Parte 4: Análisis de Correlaciones

### 🔗 Relaciones entre variables

Vamos a identificar cómo se relacionan las diferentes variables numéricas.

In [0]:
# Seleccionar variables numéricas clave para analizar correlaciones
# Elegimos las variables más relevantes para el análisis de negocio
variables_analisis = ['cantidad', 'precio_unitario_detalle', 'descuento_porcentaje', 
                      'subtotal', 'precio_unitario_producto', 'costo_unitario', 'ganancia']

# Calcular matriz de correlación
# .corr() calcula la correlación de Pearson entre todas las variables
# Valores: -1 (correlación negativa perfecta) a +1 (correlación positiva perfecta)
# 0 = sin correlación
matriz_correlacion = df_consolidado[variables_analisis].corr()

print("🔗 MATRIZ DE CORRELACIÓN")
print("=" * 80)
print(matriz_correlacion.round(3))  # Redondear a 3 decimales

# Visualizar matriz de correlación con un mapa de calor
# sns.heatmap() crea un mapa de calor con colores
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(matriz_correlacion, 
            annot=True,  # Mostrar valores numéricos en cada celda
            fmt='.2f',   # Formato con 2 decimales
            cmap='coolwarm',  # Paleta de colores: azul (negativo) a rojo (positivo)
            center=0,    # Centrar la escala de colores en 0
            square=True,  # Hacer celdas cuadradas
            linewidths=1,  # Líneas entre celdas
            ax=ax, 
            cbar_kws={'label': 'Correlación'})  # Etiqueta de la barra de color
ax.set_title('🌡️ Mapa de Calor - Correlaciones', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [0]:
# Encontrar las correlaciones más fuertes (excluyendo la diagonal)
print("💪 CORRELACIONES MÁS FUERTES")
print("=" * 80)

# Convertir matriz a series y ordenar
# .abs() toma valores absolutos (|x|) para ordenar por magnitud sin importar signo
# .unstack() convierte la matriz en una Serie de pares (var1, var2) -> correlación
correlaciones_ordenadas = matriz_correlacion.abs().unstack()
# Filtrar correlaciones perfectas (=1, que son variables consigo mismas)
# .sort_values(ascending=False) ordena de mayor a menor
correlaciones_ordenadas = correlaciones_ordenadas[correlaciones_ordenadas < 1].sort_values(ascending=False)

print("\nTop 10 correlaciones más fuertes:")
# .items() devuelve pares (clave, valor)
# enumerate(..., 1) numera comenzando en 1
for i, ((var1, var2), valor) in enumerate(correlaciones_ordenadas.head(10).items(), 1):
    print(f"{i:2d}. {var1:30s} <-> {var2:30s}: {valor:.3f}")

## Parte 5: Detección de Valores Atípicos (Outliers)

### 🔎 Identificar anomalías

Usaremos el método IQR (Rango Intercuartílico) para detectar valores atípicos.

In [0]:
# Función para detectar outliers usando el método IQR (Rango Intercuartílico)
# El método IQR es robusto y comúnmente usado en estadística
def detectar_outliers(data, columna):
    """
    Detecta valores atípicos usando el método IQR.
    
    Método:
    - Calcula Q1 (percentil 25) y Q3 (percentil 75)
    - IQR = Q3 - Q1 (rango entre el 25% y 75% de los datos)
    - Outliers: valores fuera de [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
    """
    Q1 = data[columna].quantile(0.25)  # Primer cuartil (25% de los datos están debajo)
    Q3 = data[columna].quantile(0.75)  # Tercer cuartil (75% de los datos están debajo)
    IQR = Q3 - Q1  # Rango intercuartílico (distancia entre Q1 y Q3)
    
    # Definir límites de valores "normales"
    # 1.5 * IQR es la regla estándar (método de Tukey)
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    
    # Filtrar valores fuera de los límites
    # El operador | significa OR (o)
    outliers = data[(data[columna] < limite_inferior) | (data[columna] > limite_superior)]
    
    return outliers, limite_inferior, limite_superior

# Detectar outliers en subtotal
outliers_subtotal, lim_inf, lim_sup = detectar_outliers(df_consolidado, 'subtotal')

print("🔍 DETECCIÓN DE VALORES ATÍPICOS - SUBTOTAL")
print("=" * 80)
print(f"\nLímite inferior: ${lim_inf:,.2f}")
print(f"Límite superior: ${lim_sup:,.2f}")
print(f"\nNúmero de outliers detectados: {len(outliers_subtotal)} ({len(outliers_subtotal)/len(df_consolidado)*100:.2f}%)")

if len(outliers_subtotal) > 0:
    print("\n📊 Estadísticas de outliers:")
    print(outliers_subtotal['subtotal'].describe())
    print("\n📄 Primeros 10 outliers:")
    display(outliers_subtotal[['nombre', 'categoria', 'cantidad', 'subtotal']].head(10))

In [0]:
# Visualizar outliers con scatter plot (gráfico de dispersión)
# Este gráfico muestra todos los valores y destaca los outliers en rojo
fig, ax = plt.subplots(figsize=(12, 6))

# Graficar todos los valores normales
# .scatter() crea un gráfico de puntos
# c='skyblue' define el color, alpha=0.5 la transparencia, s=10 el tamaño
ax.scatter(df_consolidado.index, df_consolidado['subtotal'], 
           c='skyblue', alpha=0.5, s=10, label='Valores normales')
# Superponer los outliers en rojo con mayor tamaño para destacarlos
ax.scatter(outliers_subtotal.index, outliers_subtotal['subtotal'], 
           c='red', alpha=0.7, s=30, label='Outliers')

# Agregar líneas horizontales para mostrar los límites
# axhline() dibuja una línea horizontal en y=valor
# linestyle='--' hace la línea punteada
ax.axhline(y=lim_sup, color='orange', linestyle='--', linewidth=2, label=f'Límite superior (${lim_sup:,.0f})')
ax.axhline(y=lim_inf, color='green', linestyle='--', linewidth=2, label=f'Límite inferior (${lim_inf:,.0f})')

ax.set_title('🎯 Detección de Valores Atípicos en Subtotales', fontsize=14, fontweight='bold')
ax.set_xlabel('Indice de registro')
ax.set_ylabel('Subtotal ($)')
ax.legend()  # Mostrar leyenda con las etiquetas
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Parte 6: Ejercicios Prácticos

### ✍️ Ejercicios para Resolver

#### **Ejercicio 1**: Análisis temporal
Crea un gráfico de líneas que muestre la evolución de las ventas totales por mes.

In [0]:
# EJERCICIO 1: Evolución temporal de ventas
# Pista: Agrupa por mes y suma los subtotales

# Tu código aquí:
# Paso 1: Agrupar por mes y sumar ventas
# .dt.to_period('M') convierte fechas a períodos mensuales (2023-01, 2023-02, etc.)
ventas_mensuales = df_consolidado.groupby(df_consolidado['fecha'].dt.to_period('M'))['subtotal'].sum()

# Paso 2: Crear gráfico de líneas
fig, ax = plt.subplots(figsize=(14, 6))
# kind='line' crea un gráfico de línea
# marker='o' agrega círculos en cada punto de datos
ventas_mensuales.plot(kind='line', ax=ax, marker='o', linewidth=2, markersize=8, color='steelblue')
ax.set_title('📈 Evolución de Ventas Mensuales', fontsize=14, fontweight='bold')
ax.set_xlabel('Mes')
ax.set_ylabel('Ventas Totales ($)')
ax.grid(True, alpha=0.3)
# Formatear eje Y para mostrar valores en millones (M)
# FuncFormatter permite personalizar cómo se muestran los valores
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))
plt.xticks(rotation=45)  # Rotar etiquetas del eje X 45 grados
plt.tight_layout()
plt.show()

# Identificar el mes con mayor facturación
# .idxmax() devuelve el índice (mes) del valor máximo
print(f"📊 Mes con mayor facturación: {ventas_mensuales.idxmax()} - ${ventas_mensuales.max():,.2f}")

## Parte 7: Perfilado Geoespacial

### 🗺️ Distribución Geográfica de Clientes

Analizaremos la distribución espacial de clientes usando índices H3 y su relación con las sucursales.

In [0]:
# Estadísticas espaciales de clientes
# H3 es un sistema de indexación hexagonal que divide el mapa en hexágonos

# Instalar librería h3 si no está disponible
%pip install h3 --quiet

import h3

print("📊 PERFILADO GEOESPACIAL")
print("=" * 60)

# Análisis de concentración de clientes por zona
# .nunique() cuenta cuántos h3_index diferentes (zonas) hay
hexagonos_unicos = df_clientes['h3_index'].nunique()
print(f"Hexágonos H3 con clientes: {hexagonos_unicos}")

# Concentración promedio = Total clientes / Zonas ocupadas
concentracion = len(df_clientes) / hexagonos_unicos
print(f"Concentración promedio: {concentracion:.2f} clientes/zona")

# Identificar outliers espaciales (clientes muy alejados de sucursales)
print("\n🔍 DISTANCIAS A SUCURSALES")
print("=" * 60)

# Calcular distancia mínima de cada cliente a cualquier sucursal
distancias = []
for _, cliente in df_clientes.iterrows():
    # Para cada cliente, calcular distancia a cada sucursal
    # h3.grid_distance() devuelve el número de hexágonos entre dos puntos
    distancias_sucursales = [
        h3.grid_distance(cliente['h3_index'], sucursal['h3_index']) 
        for _, sucursal in df_sucursales.iterrows()
    ]
    # Guardar la distancia más corta (a la sucursal más cercana)
    distancias.append(min(distancias_sucursales))

# Agregar columna de distancias al DataFrame
df_clientes['dist_min_sucursal'] = distancias

# Estadísticas de distancias
print(f"Distancia mediana a sucursal: {df_clientes['dist_min_sucursal'].median():.0f} hexágonos")
print(f"Distancia promedio: {df_clientes['dist_min_sucursal'].mean():.1f} hexágonos")

# Identificar outliers espaciales (clientes muy alejados)
# Percentil 95 = el 95% de los clientes están más cerca que este valor
percentil_95 = df_clientes['dist_min_sucursal'].quantile(0.95)
outliers_espaciales = df_clientes[df_clientes['dist_min_sucursal'] > percentil_95]
print(f"Outliers espaciales (>P95={percentil_95:.0f}): {len(outliers_espaciales)} clientes ({len(outliers_espaciales)/len(df_clientes)*100:.1f}%)")

In [0]:
# Visualizar distribución de distancias a sucursales
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histograma de distancias
# Muestra cuántos clientes hay a cada distancia
axes[0].hist(df_clientes['dist_min_sucursal'], bins=30, edgecolor='black', alpha=0.7, color='teal')
axes[0].axvline(x=percentil_95, color='red', linestyle='--', linewidth=2, label=f'Percentil 95 ({percentil_95:.0f})')
axes[0].set_title('📉 Distribución de Distancias a Sucursales', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Distancia (hexágonos H3)')
axes[0].set_ylabel('Número de clientes')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Box plot de distancias
# Muestra mediana, cuartiles y outliers
axes[1].boxplot(df_clientes['dist_min_sucursal'], vert=True)
axes[1].set_title('📦 Box Plot de Distancias', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Distancia (hexágonos H3)')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 INTERPRETACIÓN:")
print(f"   - La mayoría de clientes está a ~{df_clientes['dist_min_sucursal'].median():.0f} hexágonos de una sucursal")
print(f"   - {len(outliers_espaciales)} clientes están muy alejados (>P95)")
print(f"   - Cada hexágono ≈ ~0.1 km² (resolución 9)")

## 🎯 Resumen del TP03

### ✅ Qué aprendimos:

1. **Rutas portables**: Configuramos rutas dinámicas con detección automática del usuario
2. **Perfilado estadístico**: Generamos estadísticas descriptivas automáticas
3. **Distribución de datos**: Visualizamos histogramas y box plots
4. **Análisis de categorías**: Comparamos ventas por categoría
5. **Correlaciones**: Identificamos relaciones entre variables
6. **Detección de outliers**: Usamos el método IQR para encontrar anomalías
7. **Perfilado geoespacial**: Analizamos distribución espacial con índices H3
8. **Visualizaciones**: Creamos gráficos con matplotlib y seaborn

### 💡 Insights clave:

* **Temporales**: Identificamos el mes con mayor facturación
* **Ventas**: La mayoría de las ventas tienen subtotales moderados
* **Outliers**: Existen algunos outliers que representan ventas grandes
* **Categorías**: Ciertas categorías tienen mayor facturación
* **Correlaciones**: Hay correlaciones fuertes entre cantidad y subtotal
* **Geoespaciales**: La mayoría de clientes está cerca de una sucursal, pero hay outliers alejados

### 🚀 Próximos pasos:

En el **TP04** aprenderemos a:
* Crear dashboards interactivos
* Usar visualizaciones avanzadas con plotly
* Combinar múltiples gráficos en un dashboard
* Presentar insights de forma profesional

---

**📝 Excelente trabajo! Ahora sabes cómo perfilar y visualizar datos de forma efectiva.**